# Minecraftism - train the PyMC Bot Minecraft LLM (Kaggle, GPU T4 x2)

**Minecraftism** is a small open model fine-tuned into **the bot's brain + a Minecraft expert**: it picks the next action as
the JSON the bot already uses, chats like a player and answers questions about mobs, crafting and mechanics without a web search.

| Edition | Base | What it does |
| --- | --- | --- |
| `minecraftism` | Qwen3-4B-Instruct-2507 | text, answers straight away - **recommended for the bot** |
| `minecraftism-thinking` | Qwen3-4B-Thinking-2507 | text, thinks first (`<think>`), better in tricky spots, slower |
| `minecraftism-vl` | Qwen3-VL-4B-Instruct | also reads screenshots of the game |
| `minecraftism-vl-thinking` | Qwen3-VL-4B-Thinking | screenshots + thinking (slowest) |

**Before you press Run All**
1. On your PC: `python -m pymc_bot llm build` (add `--playtime <folder> --screenshots` for the VL editions - record with
   `/pymcbot screenshots on`, mod 0.10+) and upload the `llm_data` folder as a Kaggle dataset (skip `wiki_cache`).
2. Here: **Add Input** -> your dataset. **Settings** -> Accelerator **GPU T4 x2**, Internet **on**.
3. Optional, to publish on Hugging Face: **Add-ons -> Secrets -> Add secret** named `HF_TOKEN` (a *write* token from
   huggingface.co/settings/tokens) and tick it for this notebook. The token is only read here, never printed.
4. Pick the `EDITIONS` below. Each takes about 1-3 h; a Kaggle session stops after **12 h**, so train 1-2 per session.

When it finishes: `python -m pymc_bot llm import --hf <your-hf-name>/Minecraftism-4B-GGUF` on your PC
(or download `/kaggle/working/<edition>` from the Output tab and `python -m pymc_bot llm import <folder>`).

Made with ❤️ by codero-sus - https://github.com/codero-sus - personal, non-commercial use (PolyForm Noncommercial 1.0.0).

In [ ]:
# Which Minecraftism editions to train in this session (run in this order):
#   "minecraftism"              text, fast answers (recommended)      "minecraftism-thinking"     text, thinks first
#   "minecraftism-vl"           reads screenshots (needs pictures)    "minecraftism-vl-thinking"  pictures + thinking
# Older sizes still work too: "qwen3-4b", "qwen3-8b", "qwen3-14b" (add --split), "llama-3.2-3b", "qwen3.5-4b".
EDITIONS = ["minecraftism"]
HF_REPO = "auto"     # "auto" = <your HF name>/<edition>-GGUF (e.g. Minecraftism-4B-GGUF), "" = don't upload,
                     # "name/repo" only when training one edition
HF_PRIVATE = False   # True = private Hugging Face repos
CLEAN_AFTER_UPLOAD = True  # free /kaggle/working (20 GB) once an edition is safely on Hugging Face
EXTRA = ""           # e.g. "--epochs 2", "--max-steps 60" for a quick test, "--gguf q5_k_m", "--hub-lora"

In [ ]:
%%capture
# Unsloth's Kaggle install (same as their Qwen3 / Qwen3-VL notebooks)
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
# Hugging Face token from Kaggle's secrets (Add-ons > Secrets > HF_TOKEN). Never printed.
import os

if HF_REPO:
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded - finished editions are uploaded to Hugging Face")
    except Exception as exc:
        HF_REPO = ""
        print(f"No HF_TOKEN secret ({type(exc).__name__}) - the models stay in /kaggle/working (Output tab)")

In [ ]:
import glob
import subprocess

scripts = sorted(glob.glob("/kaggle/input/**/train_minecraft_llm.py", recursive=True))
assert scripts, "Add your PyMC Bot dataset with Add Input (it must contain train_minecraft_llm.py and train.jsonl)"
SCRIPT = scripts[0]
DATA = os.path.dirname(SCRIPT)
print("script:", SCRIPT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

READY = []
for edition in EDITIONS:
    check = subprocess.run(["python", SCRIPT, "--data", DATA, "--preset", edition, "--dry-run"], capture_output=True, text=True)
    print(f"===== {edition}\n{check.stdout[-2500:]}{check.stderr[-1500:]}")
    if check.returncode == 0:
        READY.append(edition)
    else:
        print(f"skipping {edition}: the dry run failed (see above)")
print("will train:", READY)

In [ ]:
# Training + evaluation + GGUF export (+ Hugging Face upload), one edition after another
import json
import shutil

HUB = (f'--hub-repo "{HF_REPO}"' + (" --hub-private" if HF_PRIVATE else "")) if HF_REPO else ""
for edition in READY:
    print(f"\n==================== {edition} ====================")
    !python "{SCRIPT}" --data "{DATA}" --preset {edition} {HUB} {EXTRA}
    report = f"/kaggle/working/{edition}/pymc_model.json"
    if CLEAN_AFTER_UPLOAD and os.path.exists(report) and json.load(open(report)).get("hub_repo"):
        for big in glob.glob(f"/kaggle/working/{edition}/*.gguf") + [f"/kaggle/working/{edition}/lora"]:
            shutil.rmtree(big, ignore_errors=True) if os.path.isdir(big) else os.remove(big)
        print(f"{edition}: uploaded, local copy cleaned up")

In [ ]:
for edition in READY:
    report = f"/kaggle/working/{edition}/pymc_model.json"
    if not os.path.exists(report):
        print(f"{edition}: no report - check the training output above")
        continue
    info = json.load(open(report))
    where = f"https://huggingface.co/{info['hub_repo']}" if info.get("hub_repo") else f"/kaggle/working/{edition}"
    print(f"{info.get('model_name', edition)}: loss {info.get('loss')}, eval {info.get('eval', {})}, -> {where}")

## Next
* From Hugging Face: `python -m pymc_bot llm import --hf <your-hf-name>/Minecraftism-4B-GGUF` (VL: `.../Minecraftism-VL-4B-GGUF`).
  Or download `/kaggle/working/<edition>` from the Output tab and run `python -m pymc_bot llm import <folder>`.
* The import creates the Ollama model (e.g. `minecraftism`), switches the bot to it with the short prompt and, for the VL
  editions, turns on **let the model see** (needs the optional vision packages, see docs/MINECRAFT_LLM.md).
* Compare: `python -m pymc_bot llm eval --model minecraftism` vs `--model qwen3:4b`.
* Train again later with more data: turn on **Record LLM turns** in the panel (and `/pymcbot screenshots on` in the mod),
  play, then `llm build` again.